# AbuseRing Sentinel - Synthetic Dataset Analysis (Phase 1)

Exploratory visualisation of the **synthetic** dataset produced by
`python -m src.generators.pipeline`.

**All data is synthetic.** It does not represent real customers, real payment
instruments, or real fraud rates. Nothing here measures model performance -
Phase 1 produces a validated dataset only.

Run the generator first so `data/processed/*.csv` exists:

```bash
python -m src.generators.pipeline
python -m src.validation.pipeline
```

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Make the project importable when running from notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.validation.loader import load_tables
from src.validation.distribution import build_account_features

tables = load_tables(ROOT / 'data' / 'processed')
users = tables['users']
txns = tables['transactions']
print({k: len(v) for k, v in tables.items()})

## 1. Transaction volume over time

In [ ]:
daily = txns.set_index('timestamp').resample('D').size()
ax = daily.plot(figsize=(11, 3.5), title='Daily transaction volume')
ax.set_ylabel('transactions')
plt.tight_layout()
plt.show()

## 2. Transaction amount distribution (log scale)

In [ ]:
ax = txns['amount'].clip(upper=txns['amount'].quantile(0.99)).plot(
    kind='hist', bins=60, figsize=(9, 3.5), title='Transaction amounts (99th pct clip)'
)
ax.set_xlabel('amount')
plt.tight_layout()
plt.show()

## 3. User transaction frequency

In [ ]:
per_user = txns.groupby('user_id').size()
ax = per_user.plot(kind='hist', bins=40, figsize=(9, 3.5), title='Transactions per user')
ax.set_xlabel('transactions')
plt.tight_layout()
plt.show()

## 4. Promotion usage

In [ ]:
promo_counts = txns[txns['promo_id'] != ''].groupby('promo_id').size().sort_values(ascending=False)
ax = promo_counts.head(20).plot(kind='bar', figsize=(11, 3.5), title='Top promotions by usage')
ax.set_ylabel('transactions')
plt.tight_layout()
plt.show()

## 5. Ring size distribution

In [ ]:
abuse = users[users['is_abuse_account']]
ring_sizes = abuse.groupby('ring_id').size()
ax = ring_sizes.plot(kind='hist', bins=20, figsize=(9, 3.5), title='Abuse ring sizes')
ax.set_xlabel('accounts per ring')
plt.tight_layout()
plt.show()
print(abuse.groupby('ring_type')['ring_id'].nunique())

## 6-8. Shared-infrastructure distributions (legitimate vs abuse)

In [ ]:
label = users.set_index('user_id')['is_abuse_account']
txns2 = txns.assign(is_abuse=txns['user_id'].map(label))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, col, title in zip(
    axes,
    ['device_id', 'ip_id', 'address_id'],
    ['Users per device', 'Users per IP', 'Users per address'],
):
    shared = txns2.groupby(col)['user_id'].nunique()
    shared.plot(kind='hist', bins=30, ax=ax, title=title, logy=True)
plt.tight_layout()
plt.show()

## 9. Legitimate vs abuse behavioural distributions

These should **overlap** substantially. If any single feature cleanly separates
the classes, that is a generation artifact - see `reports/leakage_audit.json`.

In [ ]:
feats = build_account_features(tables)
cols = ['txn_count', 'avg_amount', 'n_devices', 'n_ips', 'n_payments', 'promo_rate']
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), cols):
    legit = feats.loc[~feats['is_abuse_account'], col].clip(upper=feats[col].quantile(0.99))
    abuse_v = feats.loc[feats['is_abuse_account'], col].clip(upper=feats[col].quantile(0.99))
    ax.hist(legit, bins=30, alpha=0.5, density=True, label='legit')
    ax.hist(abuse_v, bins=30, alpha=0.5, density=True, label='abuse')
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()